In [32]:
import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.append(str(Path("..").resolve()))
from pathlib import Path
import re
import csv
import numpy as np
from src.parsers import _extract_complex

In [33]:
mypath = Path(r"C:\SGGGPSC\Logs\mmoin-msglog-2026-03-31.log")

In [119]:
import re
import pandas as pd

# ── 1. Parse ─────────────────────────────────────────────────────────────────

patterns = {
    "Update": re.compile(
        r"(?P<log_ts>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\.\d+ [AP]M)"
        r": (?P<level>\w+)"
        r": (?P<source>[\w.]+)"
        r": Update"
        r" : (?P<trade_time>\d{2}:\d{2} [AP]M)"
        r" : Port (?P<port>\d+)"
        r" : (?P<security>[\w.]+)"
        r" (?P<direction>\w+)"
        r" (?P<quantity>[\d,.-]+)"
        r" \((?P<user>\w+)\)"
    ),
    "Order": re.compile(
        r"(?P<log_ts>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\.\d+ [AP]M)"
        r": (?P<level>\w+)"
        r": (?P<source>[\w.]+)"
        r": Order"
        r" : (?P<trade_time>\d{2}:\d{2} [AP]M)"
        r" : (?P<action>\w+)"
        r" (?P<side>\w+)"
        r" (?P<quantity>[\d,.-]+)"
        r" (?P<security>[\w. ]+?)"
        r" FILLED"
        r" (?P<filled_qty>[\d,.-]+)"
        r" @ (?P<price>[\d.]+)"
        r" \((?P<user>\w+)\)"
    ),
}

records = []
with open(mypath, "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        matched = False
        for msg_type, pattern in patterns.items():
            m = pattern.match(line)
            if m:
                row = m.groupdict()
                row["msg_type"] = msg_type
                records.append(row)
                matched = True
                break
        if not matched:
            records.append({"raw": line, "msg_type": "UNKNOWN"})

# ── 2. Raw dataframe — never modify this ─────────────────────────────────────

df_raw = pd.DataFrame(records)

# ── 3. Cast types on a copy ───────────────────────────────────────────────────

df = df_raw.copy()

def clean_numeric(series):
    return series.str.replace(",", "", regex=False)

df["log_ts"]     = pd.to_datetime(df["log_ts"], format="%Y-%m-%d %I:%M:%S.%f %p")
df["trade_time"] = pd.to_datetime(df["trade_time"], format="%I:%M %p").dt.time
df["port"]       = pd.to_numeric(df["port"], errors="coerce")
df["quantity"]   = pd.to_numeric(clean_numeric(df["quantity"]), errors="coerce")
df["filled_qty"] = pd.to_numeric(clean_numeric(df["filled_qty"]), errors="coerce")
df["price"]      = pd.to_numeric(df["price"], errors="coerce")
df["msg_type"]   = df["msg_type"].astype("category")
df["level"]      = df["level"].astype("category")
df["source"]     = df["source"].astype("category")
df["direction"]  = df["direction"].astype("category")
df["action"]     = df["action"].astype("category")
df["side"]       = df["side"].astype("category")
df["user"]       = df["user"].astype("category")

df = df.sort_values("log_ts").reset_index(drop=True)
df["elapsed_ms"] = df["log_ts"].diff().dt.total_seconds()

log_ts        155
level         155
source        155
trade_time    155
port          150
security      155
direction     150
quantity      155
user          155
msg_type      155
raw             0
action          5
side            5
filled_qty      5
price           5
elapsed_ms    154
dtype: int64